In [5]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors


# -------------------------
# PDF compatibility
# -------------------------
# Avoid "Insufficient data for an image" in some PDF readers.
mpl.rcParams["pdf.compression"] = 0


# -------------------------
# Style
# -------------------------
sns.set_theme(style="white", context="paper")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "axes.linewidth": 0.8,
    "font.family": "Times New Roman",
})

LINE_COLOR = "#2C3E50"


# -------------------------
# Project paths
# -------------------------
# The Jupyter Notebook working directory should be:
# Code/Figures Python
project_dir = Path.cwd()

data_dir = project_dir / "Data"
results_dir = project_dir / "Results"


# -------------------------
# Input and output
# -------------------------
file_xlsx = (
    data_dir
    / "MSA_PFF_GWAS_Enrichment_PD_MSA.xlsx"
)

sheet_name = "PD_GWAS"

out_dir = (
    results_dir
    / "MSA_PFF_GWAS_Enrichment_PD_MSA"
    / "PD_GWAS"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)


# -------------------------
# Read and sanitize
# -------------------------
df = pd.read_excel(
    file_xlsx,
    sheet_name=sheet_name,
    header=0
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

REQ = [
    "top.N",
    "Enrichment_Score",
    "FDR_P_Val",
    "list2"
]

missing = [
    c
    for c in REQ
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns in sheet '{sheet_name}': {missing}"
    )

df["top.N"] = pd.to_numeric(
    df["top.N"],
    errors="coerce"
)

df["Enrichment_Score"] = pd.to_numeric(
    df["Enrichment_Score"],
    errors="coerce"
)

df["FDR_P_Val"] = pd.to_numeric(
    df["FDR_P_Val"],
    errors="coerce"
)

df["list2"] = (
    df["list2"]
    .astype(str)
    .str.strip()
)

df = df.dropna(
    subset=[
        "top.N",
        "Enrichment_Score",
        "FDR_P_Val",
        "list2"
    ]
).copy()


# -------------------------
# Color metric
# -------------------------
# MATLAB-style transformation:
# -log10(FDR) + 1
FLOOR = 1e-300

df["FDR_P_Val"] = df["FDR_P_Val"].clip(
    lower=FLOOR
)

df["color_val"] = (
    -np.log10(
        df["FDR_P_Val"].to_numpy(dtype=float)
    )
    + 1.0
)


# -------------------------
# Significance
# -------------------------
alpha = 0.05

df["is_sig"] = (
    df["FDR_P_Val"] <= alpha
)


# -------------------------
# Axis order
# -------------------------
# GCI labels are replaced by MSA.
x_order = [
    "PFF_map.rep",
    "PFF_whole",
    "MSA_map.rep",
    "MSA_whole"
]

df = df[
    df["list2"].isin(x_order)
].copy()

df["list2"] = pd.Categorical(
    df["list2"],
    categories=x_order,
    ordered=True
)

y_order = [
    100,
    200,
    300,
    400
]

df = df[
    df["top.N"].isin(y_order)
].copy()

df["top.N"] = pd.Categorical(
    df["top.N"].astype(int),
    categories=y_order,
    ordered=True
)


# -------------------------
# Colormap
# -------------------------
muted_red = mcolors.LinearSegmentedColormap.from_list(
    "muted_red",
    [
        (0.98, 0.94, 0.94),
        (0.96, 0.78, 0.76),
        (0.92, 0.55, 0.50),
        (0.80, 0.22, 0.18),
        (0.55, 0.02, 0.06),
    ],
    N=256
)

cmap = muted_red


# Use significant values for the color range when available.
sig_vals = df.loc[
    df["is_sig"],
    "color_val"
].to_numpy(dtype=float)

if sig_vals.size == 0:
    sig_vals = df[
        "color_val"
    ].to_numpy(dtype=float)

vmin = max(
    1.0,
    float(np.nanmin(sig_vals))
)

vmax = float(
    np.nanpercentile(
        sig_vals,
        95
    )
)

if vmax <= vmin:
    vmax = vmin + 1e-6

norm = mpl.colors.Normalize(
    vmin=vmin,
    vmax=vmax
)


# -------------------------
# Bubble size mapping
# -------------------------
s_raw = df[
    "Enrichment_Score"
].to_numpy(dtype=float)

s_min = float(
    np.nanpercentile(
        s_raw,
        5
    )
)

s_max = float(
    np.nanpercentile(
        s_raw,
        95
    )
)

if s_max <= s_min:
    s_max = s_min + 1e-6

AREA_MIN = 80
AREA_MAX = 900


def size_to_area(s):
    """
    Convert Fold Enrichment values into scatter-marker areas.
    """
    s = np.asarray(
        s,
        dtype=float
    )

    s = np.clip(
        s,
        s_min,
        s_max
    )

    return (
        AREA_MIN
        + (s - s_min)
        / (s_max - s_min)
        * (AREA_MAX - AREA_MIN)
    )


df["area"] = size_to_area(
    df["Enrichment_Score"].to_numpy(dtype=float)
)


# -------------------------
# Fixed size-legend values for PD
# -------------------------
ref_sizes = np.array(
    [1.4, 2.1],
    dtype=float
)

ref_areas = size_to_area(
    ref_sizes
)


# -------------------------
# Layout
# -------------------------
fig = plt.figure(
    figsize=(8.6, 4.4)
)

gs = fig.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[
        1.0,
        0.70
    ],
    wspace=0.25
)

ax = fig.add_subplot(
    gs[0, 0]
)

axR = fig.add_subplot(
    gs[0, 1]
)

axR.set_axis_off()


n_cols = len(x_order)
n_rows = len(y_order)

ax.set_xlim(
    0.5,
    n_cols + 0.5
)

ax.set_ylim(
    0.5,
    n_rows + 0.5
)

ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_facecolor("white")

ax.set_xticks(
    range(1, n_cols + 1)
)

ax.set_xticklabels(
    x_order,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    range(1, n_rows + 1)
)

ax.set_yticklabels(
    [str(y) for y in y_order]
)

ax.tick_params(
    axis="both",
    direction="out",
    length=3,
    width=0.8
)


# -------------------------
# Grid
# -------------------------
for y in np.arange(
    0.5,
    n_rows + 1.5,
    1.0
):
    ax.plot(
        [0.5, n_cols + 0.5],
        [y, y],
        color=[0.86, 0.86, 0.86],
        lw=0.9,
        zorder=0
    )

for x in np.arange(
    0.5,
    n_cols + 1.5,
    1.0
):
    ax.plot(
        [x, x],
        [0.5, n_rows + 0.5],
        color=[0.86, 0.86, 0.86],
        lw=0.9,
        zorder=0
    )

for spine in ax.spines.values():
    spine.set_linewidth(0.8)


# -------------------------
# Map labels to coordinates
# -------------------------
x_map = {
    name: i + 1
    for i, name in enumerate(x_order)
}

y_map = {
    value: i + 1
    for i, value in enumerate(y_order)
}

GREY = (
    0.72,
    0.72,
    0.72
)


# -------------------------
# Draw bubbles
# -------------------------
for _, row in df.iterrows():
    x = x_map[
        str(row["list2"])
    ]

    y = y_map[
        int(row["top.N"])
    ]

    area = float(
        row["area"]
    )

    if bool(row["is_sig"]):
        face = cmap(
            norm(
                float(row["color_val"])
            )
        )

        edge = "black"
        linewidth = 1.6

    else:
        face = GREY
        edge = "none"
        linewidth = 0.0

    ax.scatter(
        x,
        y,
        s=area,
        c=[face],
        edgecolors=edge,
        linewidths=linewidth,
        zorder=3
    )


# -------------------------
# Right panel
# -------------------------
bbox = axR.get_position()

cax = fig.add_axes([
    bbox.x0 + 0.05 * bbox.width,
    bbox.y0 + 0.58 * bbox.height,
    0.18 * bbox.width,
    0.34 * bbox.height
])

sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    cax=cax,
    orientation="vertical"
)

cbar.outline.set_linewidth(0.8)

cbar.ax.tick_params(
    direction="out",
    length=3,
    width=0.8
)

cbar.ax.yaxis.set_major_formatter(
    mpl.ticker.FormatStrFormatter("%.1f")
)

fig.text(
    cax.get_position().x0,
    cax.get_position().y1 + 0.02,
    r"$-\log_{10}(\mathrm{FDR}) + 1$",
    ha="left",
    va="bottom",
    fontsize=14
)


# -------------------------
# Fixed PD size legend
# -------------------------
axR.set_xlim(0, 1)
axR.set_ylim(0, 1)
axR.set_aspect("equal")

axR.text(
    0.05,
    0.46,
    "Fold Enrichment",
    ha="left",
    va="bottom",
    fontsize=14
)

legend_y = [
    0.29,
    0.12
]

for y0, size_value, area_value in zip(
    legend_y,
    ref_sizes,
    ref_areas
):
    axR.scatter(
        0.18,
        y0,
        s=area_value,
        c="black",
        edgecolors="none"
    )

    axR.text(
        0.36,
        y0,
        f"{size_value:.1f}",
        ha="left",
        va="center",
        fontsize=12
    )


# -------------------------
# Save
# -------------------------
save_prefix = "PD_GWAS_enrichment_bubble"

pdf_path = (
    out_dir
    / f"{save_prefix}.pdf"
)

png_path = (
    out_dir
    / f"{save_prefix}.png"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print("Saved:")
print(pdf_path)
print(png_path)

Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_GWAS_Enrichment_PD_MSA\PD_GWAS\PD_GWAS_enrichment_bubble.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_GWAS_Enrichment_PD_MSA\PD_GWAS\PD_GWAS_enrichment_bubble.png


In [6]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors


# -------------------------
# PDF compatibility
# -------------------------
mpl.rcParams["pdf.compression"] = 0


# -------------------------
# Style
# -------------------------
sns.set_theme(style="white", context="paper")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "axes.linewidth": 0.8,
    "font.family": "Times New Roman",
})

LINE_COLOR = "#2C3E50"


# -------------------------
# Project paths
# -------------------------
project_dir = Path.cwd()

data_dir = project_dir / "Data"
results_dir = project_dir / "Results"


# -------------------------
# Input and output
# -------------------------
file_xlsx = (
    data_dir
    / "MSA_PFF_GWAS_Enrichment_PD_MSA.xlsx"
)

sheet_name = "MSA_GWAS"

out_dir = (
    results_dir
    / "MSA_PFF_GWAS_Enrichment_PD_MSA"
    / "MSA_GWAS"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)


# -------------------------
# Read and sanitize
# -------------------------
df = pd.read_excel(
    file_xlsx,
    sheet_name=sheet_name,
    header=0
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

REQ = [
    "top",
    "group",
    "FDR_P_Val",
    "Fold_Enrichment"
]

missing = [
    c
    for c in REQ
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns in sheet '{sheet_name}': {missing}"
    )

df["top"] = pd.to_numeric(
    df["top"],
    errors="coerce"
)

df["FDR_P_Val"] = pd.to_numeric(
    df["FDR_P_Val"],
    errors="coerce"
)

df["Fold_Enrichment"] = pd.to_numeric(
    df["Fold_Enrichment"],
    errors="coerce"
)

df["group"] = (
    df["group"]
    .astype(str)
    .str.strip()
)

df = df.dropna(
    subset=[
        "top",
        "group",
        "FDR_P_Val",
        "Fold_Enrichment"
    ]
).copy()


# -------------------------
# Color metric
# -------------------------
FLOOR = 1e-300

df["FDR_P_Val"] = df["FDR_P_Val"].clip(
    lower=FLOOR
)

df["color_val"] = (
    -np.log10(
        df["FDR_P_Val"].to_numpy(dtype=float)
    )
    + 1.0
)


# -------------------------
# Significance
# -------------------------
alpha = 0.05

df["is_sig"] = (
    df["FDR_P_Val"] <= alpha
)


# -------------------------
# Axis order
# -------------------------
x_order = [
    "PFF",
    "MSA"
]

df = df[
    df["group"].isin(x_order)
].copy()

df["group"] = pd.Categorical(
    df["group"],
    categories=x_order,
    ordered=True
)

y_order = [
    100,
    200,
    300,
    400
]

df = df[
    df["top"].isin(y_order)
].copy()

df["top"] = pd.Categorical(
    df["top"].astype(int),
    categories=y_order,
    ordered=True
)


# -------------------------
# Colormap
# -------------------------
muted_red = mcolors.LinearSegmentedColormap.from_list(
    "muted_red",
    [
        (0.98, 0.94, 0.94),
        (0.96, 0.78, 0.76),
        (0.92, 0.55, 0.50),
        (0.80, 0.22, 0.18),
        (0.55, 0.02, 0.06),
    ],
    N=256
)

cmap = muted_red

all_vals = df[
    "color_val"
].to_numpy(dtype=float)

vmin = float(
    np.nanmin(all_vals)
)

vmax = float(
    np.nanmax(all_vals)
)

if vmax <= vmin:
    vmax = vmin + 1e-6

norm = mpl.colors.Normalize(
    vmin=vmin,
    vmax=vmax
)


# -------------------------
# Bubble size mapping
# -------------------------
s_raw = df[
    "Fold_Enrichment"
].to_numpy(dtype=float)

s_min = float(
    np.nanpercentile(
        s_raw,
        5
    )
)

s_max = float(
    np.nanpercentile(
        s_raw,
        95
    )
)

if s_max <= s_min:
    s_max = s_min + 1e-6

AREA_MIN = 90
AREA_MAX = 1100


def size_to_area(s):
    """
    Convert Fold Enrichment values into scatter-marker areas.
    """
    s = np.asarray(
        s,
        dtype=float
    )

    s = np.clip(
        s,
        s_min,
        s_max
    )

    return (
        AREA_MIN
        + (s - s_min)
        / (s_max - s_min)
        * (AREA_MAX - AREA_MIN)
    )


df["area"] = size_to_area(
    df["Fold_Enrichment"].to_numpy(dtype=float)
)


# -------------------------
# Fixed size-legend values for MSA
# -------------------------
ref_sizes = np.array(
    [
        0.87,
        0.98,
        1.42
    ],
    dtype=float
)

ref_areas = size_to_area(
    ref_sizes
)


# -------------------------
# Layout
# -------------------------
fig = plt.figure(
    figsize=(6.9, 4.4)
)

gs = fig.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[
        1.0,
        0.78
    ],
    wspace=0.28
)

ax = fig.add_subplot(
    gs[0, 0]
)

axR = fig.add_subplot(
    gs[0, 1]
)

axR.set_axis_off()


n_cols = len(x_order)
n_rows = len(y_order)

ax.set_xlim(
    0.5,
    n_cols + 0.5
)

ax.set_ylim(
    0.5,
    n_rows + 0.5
)

ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_facecolor("white")

ax.set_xticks(
    range(1, n_cols + 1)
)

ax.set_xticklabels(
    x_order,
    rotation=0,
    ha="center"
)

ax.set_yticks(
    range(1, n_rows + 1)
)

ax.set_yticklabels(
    [str(y) for y in y_order]
)

ax.tick_params(
    axis="both",
    direction="out",
    length=3,
    width=0.8
)


# -------------------------
# Grid
# -------------------------
for y in np.arange(
    0.5,
    n_rows + 1.5,
    1.0
):
    ax.plot(
        [0.5, n_cols + 0.5],
        [y, y],
        color=[0.86, 0.86, 0.86],
        lw=0.9,
        zorder=0
    )

for x in np.arange(
    0.5,
    n_cols + 1.5,
    1.0
):
    ax.plot(
        [x, x],
        [0.5, n_rows + 0.5],
        color=[0.86, 0.86, 0.86],
        lw=0.9,
        zorder=0
    )

for spine in ax.spines.values():
    spine.set_linewidth(0.8)


# -------------------------
# Grid coordinates
# -------------------------
x_map = {
    name: i + 1
    for i, name in enumerate(x_order)
}

y_map = {
    value: i + 1
    for i, value in enumerate(y_order)
}


# -------------------------
# Draw bubbles
# -------------------------
for _, row in df.iterrows():
    x = x_map[
        str(row["group"])
    ]

    y = y_map[
        int(row["top"])
    ]

    area = float(
        row["area"]
    )

    face = cmap(
        norm(
            float(row["color_val"])
        )
    )

    # Only significant points receive a black outline.
    if bool(row["is_sig"]):
        edge = "black"
        linewidth = 1.6
    else:
        edge = "none"
        linewidth = 0.0

    ax.scatter(
        x,
        y,
        s=area,
        c=[face],
        edgecolors=edge,
        linewidths=linewidth,
        zorder=3
    )


# -------------------------
# Right panel
# -------------------------
bbox = axR.get_position()

cax = fig.add_axes([
    bbox.x0 + 0.05 * bbox.width,
    bbox.y0 + 0.58 * bbox.height,
    0.20 * bbox.width,
    0.34 * bbox.height
])

sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    cax=cax,
    orientation="vertical"
)

cbar.outline.set_linewidth(0.8)

cbar.ax.tick_params(
    direction="out",
    length=3,
    width=0.8
)

cbar.ax.yaxis.set_major_formatter(
    mpl.ticker.FormatStrFormatter("%.1f")
)

fig.text(
    cax.get_position().x0,
    cax.get_position().y1 + 0.02,
    r"$-\log_{10}(\mathrm{FDR}) + 1$",
    ha="left",
    va="bottom",
    fontsize=14
)


# -------------------------
# Fixed MSA size legend
# -------------------------
axR.set_xlim(0, 1)
axR.set_ylim(0, 1)
axR.set_aspect("equal")

axR.text(
    0.05,
    0.46,
    "Fold Enrichment",
    ha="left",
    va="bottom",
    fontsize=14
)

legend_y = [
    0.32,
    0.20,
    0.08
]

for y0, size_value, area_value in zip(
    legend_y,
    ref_sizes,
    ref_areas
):
    axR.scatter(
        0.18,
        y0,
        s=area_value,
        c="black",
        edgecolors="none"
    )

    axR.text(
        0.36,
        y0,
        f"{size_value:.2f}",
        ha="left",
        va="center",
        fontsize=12
    )


# -------------------------
# Save
# -------------------------
save_prefix = "MSA_GWAS_FisherTest_bubble"

pdf_path = (
    out_dir
    / f"{save_prefix}.pdf"
)

png_path = (
    out_dir
    / f"{save_prefix}.png"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print("Saved:")
print(pdf_path)
print(png_path)

Saved:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_GWAS_Enrichment_PD_MSA\MSA_GWAS\MSA_GWAS_FisherTest_bubble.pdf
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_GWAS_Enrichment_PD_MSA\MSA_GWAS\MSA_GWAS_FisherTest_bubble.png
